In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/katla@mail.bradley.edu/ML/Data_Engineering/Setup/utils

In [0]:
dbutils.widgets.text("catalog", "ecommerce", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

In [0]:
catalog= dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path=f'/Workspace/Users/katla@mail.bradley.edu/ML/Data_Engineering/Data/{data_source}'
print(base_path)

In [0]:
df_bronze= spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

In [0]:
df_bronze = df_bronze.withColumn(
    "product_category_name",
    F.regexp_replace(F.col("product_category_name"), "_", " ")
)

In [0]:
df_bronze=df_bronze.withColumn(
    'category',
    F.initcap(F.col('product_category_name'))
)

In [0]:
df_silver=df_bronze.drop('product_category_name')
df_silver.display(10)


In [0]:
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
df_silver= spark.sql(f'select * from {catalog}.{silver_schema}.{data_source}')

df_gold= df_silver.select('product_id','category', 'product_weight_g')

In [0]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")